# AII 600. Lab 4

Due before class, 23 September 2026.

Upload this notebook on Canvas, with every code cell executed and the output left in. Credit is on-time submission. I do not mark the lab.

Python kernel. R is allowed: switch to an R kernel and rewrite the code cells. Ask the tutor to translate `check`.

Two decisions, then the three named families, then the 2-by-2 of model and data. Method of moments is the estimator: equate $\bar x$ to $E(X)$, and if you need a scale, equate the second moment to $\mathrm{Var}(X)$. The denominator is $n$, not $n-1$. `numpy.var` does that; `pandas.Series.var` does not. Work each numeric question on paper first, then type the expression. Written cells have no hashed check. After you write them, ask your tutor to check the written answers.

The notes use `dbinom`, `dpois`, `pnorm`. In Python that is `scipy.stats.binom`, `poisson`, and `norm`. `norm.cdf` is `pnorm`. The notes write $N(\mu,\sigma^2)$. scipy takes `loc=μ` and `scale=σ`, the standard deviation, not the variance. The two-binomial cell is `math.comb` first, so the pmf is still on your cheat sheet.

The test cell under each answer is the grader. It does not contain the number. Fill the answer cell, run it, then run the test cell. If it prints `ok`, that name is right to six decimals. If it says a name is not right, rework that quantity; the cell will not tell you the value. Do not paste a three-decimal number from the slides. The midterm has no test cell.

These are not a substitute for a cheat sheet. After you finish, copy the formulas you actually used onto one.


## How an exercise is built

Run the next cell once. It defines `check`. Each numeric exercise then has two code cells. The first is yours: replace every `None`. The second calls `check`. A line that prints `ok` passed. Leave the output in the notebook.

Do not delete the variable names. You can add scratch cells above an answer cell if you want to compute in public. A written cell has no `check`. Write it, then ask your tutor to read it and say whether the reasoning is right.

You will write a secretary simulator, compute $E$ and $\mathrm{Var}$ of two oil drills, cancel $p$ in a pair of binomials, and fit Poisson and Normal by method of moments on real files. `numpy`, `matplotlib`, `scipy.stats`, and `pandas` are allowed after the `math.comb` cell. Ask the tutor for syntax; do the probability yourself.


In [ ]:
from hashlib import sha256

def check(name, value, expected, ndigits=6):
    if value is None:
        raise AssertionError(f"replace None for {name}")
    s = f"{round(float(value), ndigits):.{ndigits}f}"
    got = sha256(f"aii600-lab4|{name}|{s}".encode()).hexdigest()[:16]
    assert got == expected, f"{name} is not right"
    print("ok ", name)


## Exercise 1: secretary, write the simulator

You will see $T=100$ candidates, one at a time. After you pass, you cannot go back. You want the overall best. The lecture plot used $T=1000$; the shape is the same and this one finishes in a few seconds.

Scores are a random permutation of $1,\ldots,T$, with $T$ the best and $1$ the worst. The $r$-rule: skip the first $r$, then take the next candidate who is best so far. If nobody after $r$ beats the prefix, take the last. If $r=0$, take the first.

At least $8{,}000$ Monte Carlo lists. For each $r=0,1,\ldots,T-1$ record $P(\text{pick the overall best})$ and the mean and variance of the *score* you hired (the value in $1,\ldots,T$, not a 0/1 win).

Write the simulator in the next cell. The starter is imports, $T$, and `plot_secretary`, which already draws the $T/e$ and $1/e$ lines and labels the axes `Number screened` and `P(pick the best)`. You still need `screened` ($0,1,\ldots,T-1$) and `p_pick_best` (length $T$). Exact success: `p_best_rstar`, `p_best_r1`. From the simulator: `p_best_rstar_sim`, `p_best_r1_sim`, `e_score_rstar`, `var_score_rstar`, `e_score_r1`, `var_score_r1`.

1. After the simulator, call `plot_secretary(screened, p_pick_best, T)`.
2. Exact success probability from the lecture sum, not from the Monte Carlo,
$$
P(\text{success})=\frac{r}{T}\sum_{b=r}^{T-1}\frac{1}{b}
$$
for $r\ge 1$. Hash those names at $r=\mathrm{round}(T/e)$ and at $r=1$.
3. From the simulator: the same two success rates, plus $E(\text{score})$ and $\mathrm{Var}(\text{score})$. The simulation has to be close; it is not hashed.

Do not paste $0.368$ from the slide.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

T = 100
n_mc = 10000
rng = np.random.default_rng()
r_star = round(T / np.e)


def plot_secretary(screened, p_pick_best, T):
    plt.plot(screened, p_pick_best, color="#086cc4", linewidth=1.5)
    plt.axvline(T / np.e, color="#b42318", linewidth=0.8)
    plt.axhline(1 / np.e, color="#b42318", linewidth=0.8, linestyle="--")
    plt.xlabel("Number screened")
    plt.ylabel("P(pick the best)")
    plt.show()


# write the r-rule simulator, then
# plot_secretary(screened, p_pick_best, T)


In [ ]:
check("p_best_rstar", p_best_rstar, "82f58f7afbcf2ab3")
check("p_best_r1", p_best_r1, "8c6c2af2f4cf19d2")
print("ok  exercise 1 exact success")
assert n_mc >= 8000
assert r_star == 37
assert p_best_rstar_sim is not None and p_best_r1_sim is not None
assert abs(p_best_rstar_sim - p_best_rstar) < 0.03, "Monte Carlo P(best) at r_star should match the sum"
assert abs(p_best_r1_sim - p_best_r1) < 0.02, "Monte Carlo P(best) at r=1 should match the sum"
assert e_score_rstar is not None and var_score_rstar is not None
assert e_score_r1 is not None and var_score_r1 is not None
assert abs(e_score_rstar - 81.22555) < 2.5, "E(score) at r_star is off; 1 is worst, T is best"
assert abs(var_score_rstar - 859.5063771975) < 150, "Var(score) at r_star is off"
assert abs(e_score_r1 - 74.91105) < 2.5, "E(score) at r=1 is off"
assert abs(var_score_r1 - 491.1912378975) < 150, "Var(score) at r=1 is off"
import numpy as np
screened = np.asarray(screened)
p_pick_best = np.asarray(p_pick_best, dtype=float)
assert len(screened) == T and len(p_pick_best) == T
assert np.allclose(screened, np.arange(T)), "number screened is r = 0, 1, ..., T-1"
assert abs(float(p_pick_best[0]) - 1 / T) < 0.01, "r=0 takes the first and wins with probability 1/T"
assert abs(float(p_pick_best[r_star]) - p_best_rstar) < 0.03
print("ok  exercise 1 simulation")


The $r$ that maximises $P(\text{pick the best})$ is not the $r$ that maximises $E(\text{score})$. Which way has $E(\text{score})$ already peaked, and why is that a different decision? Four to six sentences.

YOUR ANSWER HERE


## Exercise 2: oil drills, expectation and variance

Book Chapter 1, Expectation and Strategy. An oil company can use a standard drill or a more expensive horizontal drill. Chance of a small / moderate / large find: $0.2$, $0.5$, $0.3$. Payoffs in \$millions:

| | small | moderate | large |
|--|------:|---------:|------:|
| Standard | 20 | 30 | 40 |
| Horizontal | $-20$ | 40 | 80 |

1. $E$ and $\mathrm{Var}$ of payoff for each drill. $\mathrm{Var}(X)=E(X^2)-[E(X)]^2$.
2. With a perfect geological study you see the size of the find before you choose the drill. The most you should pay for that study: expected payoff with the study, minus the better of the two means from (1). This quantity is the value of information; week 7 gives it that name.

Do not paste $31$ or $40$ from the book.


In [ ]:
e_standard = None
var_standard = None
e_horizontal = None
var_horizontal = None
wtp_study = None        # most you pay for a perfect study
e_standard, var_standard, e_horizontal, var_horizontal, wtp_study


In [ ]:
check("e_standard", e_standard, "f655bbdfb21398f5")
check("var_standard", var_standard, "df50ba225c3a712e")
check("e_horizontal", e_horizontal, "50ab8d777e664029")
check("var_horizontal", var_horizontal, "33a70f7b0e3aaac0")
check("wtp_study", wtp_study, "4c4a068c6ac2b743")
print("ok  exercise 2 oil")


Which drill maximises expected payoff? If a $\$20$ million loss is a problem for the firm, which do you choose, and what assumption about "maximise $E(X)$" just failed? For the study: what did you assume about it? Four to six sentences.

YOUR ANSWER HERE


## Exercise 3: two binomials

Let $X$ and $Y$ be independent $\mathrm{Binomial}(n,p)$. Show
$$
P(X=k\mid X+Y=m)=\frac{\binom{n}{k}\binom{n}{m-k}}{\binom{2n}{m}},\qquad k=0,1,\ldots,\min(m,n).
$$
Write the derivation in this cell before you compute. Start from $P(X=k,Y=m-k)/P(X+Y=m)$. Name the distribution of $X+Y$. Show that $p$ and $q=1-p$ cancel. Do not paste a book display.

YOUR ANSWER HERE


Now compute. $n=12$, $m=9$, $k=4$. `math.comb` only on this cell. Do not call `scipy.stats.binom`.

1. $P(X=4\mid X+Y=9)$ from the combo formula.
2. $E(X\mid X+Y=9)$ and $\mathrm{Var}(X\mid X+Y=9)$. Symmetry gives the mean. For the variance, sum $k^2 P(X=k\mid X+Y=9)$ or use the hypergeometric with population $2n$, $n$ of type $X$, $m$ draws.


In [ ]:
from math import comb

n, m, k = 12, 9, 4
p_cond = None
e_cond = None
var_cond = None
p_cond, e_cond, var_cond


In [ ]:
check("p_cond", p_cond, "c5e3a5067c98559a")
check("e_cond", e_cond, "f2404a7845a3c6ec")
check("var_cond", var_cond, "5c2c5bbe3593c4d6")
print("ok  exercise 3 comb")


Same probability from the binomial pmf ratio $P(X=k)P(Y=m-k)/P(X+Y=m)$, once at $p=0.2$ and once at $p=0.8$. A library pmf is allowed here. The two numbers should match each other and match `p_cond`.


In [ ]:
from scipy.stats import binom

p_pmf_02 = None
p_pmf_08 = None
p_pmf_02, p_pmf_08


In [ ]:
check("p_pmf_02", p_pmf_02, "fa197f0b56084dad")
check("p_pmf_08", p_pmf_08, "ff3bf985c4c97107")
print("ok  exercise 3 pmf")


`satgpa.csv`, in the same folder as this notebook (or [the course copy](https://vsokolov.org/courses/files/600/satgpa.csv)). 1,000 students. A success is `sat_sum >= 120`. Let $X$ be the number of successes in the first 500 rows and $Y$ the number in the last 500. Model both as $\mathrm{Binomial}(500,p)$.

4. Method of moments: $\hat p=(X+Y)/1000$. Hash $\hat p$, the observed $X$, and $P(X=X_{\mathrm{obs}}\mid X+Y=m)$ from the combo formula, still `math.comb`.


In [ ]:
from pathlib import Path
from math import comb
import pandas as pd

path = Path("satgpa.csv")
if not path.exists():
    path = "https://vsokolov.org/courses/files/600/satgpa.csv"
sat = pd.read_csv(path)

p_hat = None
x_obs = None
p_split = None     # P(X = x_obs | X+Y = x_obs+y_obs)
p_hat, x_obs, p_split


In [ ]:
check("p_hat", p_hat, "386dfd926e04ab83")
check("x_obs", x_obs, "6b65da565b774602")
check("p_split", p_split, "0614a9d806835852")
print("ok  exercise 3 sat binomial")


You did not need $\hat p$ to split the total; you needed it for the marginal. If the csv is ordered by a hidden type, the two halves are not the same $p$ and the identity's assumptions fail. How is that next hour's "bad data"? Three to five sentences.

YOUR ANSWER HERE


## Exercise 4: good data, good model (Berkson Poisson)

Exercises 4 to 7 are one 2-by-2. The same two questions each time: is the *sample* the process you will decide for, and is the *family* the right law?

| | good model | bad model |
|---|---|---|
| good data | 4. Berkson alpha particles | 5. 1987 crash |
| bad data | 6. SAT selected by GPA | 7. the same rows as Poisson |

Watch one thing as you go. On the model axis the data argues back: in 4 the overlay tracks, in 5 the $z$ is absurd. On the data axis nothing argues back. Exercise 6 will look fine from the inside.

This is cell (good data, good model). Berkson (1966), americium-241, $N=1207$ intervals of 10 seconds. `berkson.csv`, in the same folder (or [the course copy](https://vsokolov.org/courses/files/600/berkson.csv)). Columns `n` and `Observed`. The row `n=1` lumps counts 0, 1, and 2, as the lecture file does. Use the `n` column as the count, lumped row included. Your $\hat\lambda$ is per 10-second interval. The lecture divides by 10 to report a rate per second; do not copy that step.

Method of moments for Poisson: $\hat\lambda=\bar x=\sum n\cdot\mathrm{Observed}_n / N$.

1. $\hat\lambda$ and the expected count at $k=8$, namely $N\,e^{-\hat\lambda}\hat\lambda^8/8!$. Do not paste $8.35$.
2. Overlay observed counts against $N$ times the Poisson pmf at each `n` in the file. The `n=1` row holds counts 0, 1 and 2, so its expected height is $N\,P(X\le 2)$, not $N\,P(X=1)$. The lecture lumps the same three. Label the x-axis `count`.


In [ ]:
from pathlib import Path
import pandas as pd

path = Path("berkson.csv")
if not path.exists():
    path = "https://vsokolov.org/courses/files/600/berkson.csv"
berk = pd.read_csv(path)

lam_hat = None
expected_8 = None
# overlay observed vs Poisson expected; xlabel count
lam_hat, expected_8


In [ ]:
check("lam_hat", lam_hat, "cde3872f449dbf63")
check("expected_8", expected_8, "97d62b6462fc28cd")
print("ok  exercise 4 berkson")


Name the cell: good data and a good model. Your overlay is the evidence. Why are the NBS intervals the process you care about, and why is Poisson the right family? The lumped 0--2 row is a small file defect, not a wrong population. Three to five sentences.

YOUR ANSWER HERE


## Exercise 5: good data, bad model (1987 crash)

This is cell (good data, bad model). October 1987 the S\&P 500 dropped $-21.76\%$. `sp500_precrash.csv` is real monthly returns through September 1987, in the same folder (or [the course copy](https://vsokolov.org/courses/files/600/sp500_precrash.csv)). Column `monthly_return`. The crash month is not in the file.

Model: $R\sim N(\mu,\sigma^2)$. Method of moments: $\hat\mu=\bar r$ and $\hat\sigma^2=\frac1n\sum(r_i-\bar r)^2$.

1. $n$, $\hat\mu$, $\hat\sigma^2$.
2. $z$ for a return of $-0.2176$ under that fitted Normal, $P(R\le -0.2176)$, and $\log_{10}$ of that probability. The probability is smaller than six decimal places can hold, so the check reads the $\log_{10}$.
3. Histogram of the monthly returns with the fitted Normal density overlaid, and mark $-0.2176$ on the axis.

Item 3 is the exercise, not decoration. The fitted Normal has to put the crash somewhere, and you need to see where it put it.

Do not paste $-5.34$ or $4.3\%$ from the slide.


In [ ]:
from pathlib import Path
import pandas as pd

path = Path("sp500_precrash.csv")
if not path.exists():
    path = "https://vsokolov.org/courses/files/600/sp500_precrash.csv"
sp = pd.read_csv(path)

n_months = None
mu_hat = None
sigma2_hat = None
z_crash = None
p_crash = None
log10_p_crash = None    # log10(p_crash); the probability underflows six decimals
n_months, mu_hat, sigma2_hat, z_crash, p_crash, log10_p_crash


In [ ]:
check("n_months", n_months, "8f278554eceb3830")
check("mu_hat", mu_hat, "94ba8bdcbb9207a5")
check("sigma2_hat", sigma2_hat, "0aa1f795ea7a6008")
check("z_crash", z_crash, "e31aaa5f7d999083")
check("log10_p_crash", log10_p_crash, "3b9aa6bb7953f65e")
print("ok  exercise 5 crash")


The months are a real sample of the market you will decide for. The Normal is the wrong law. Same picture as tonight's orange curve. Your histogram shows where the fitted Normal put the crash. What did the model call the crash, and what did a 5-sigma label actually measure? Four to six sentences.

YOUR ANSWER HERE


## Exercise 6: good model, bad data (shift / healthy-user)

This is cell (good model, bad data). Same `satgpa.csv`, but now the *raw* `sat_sum` as Normal, not the Bernoulli from Exercise 3. The incoming class is the full 1,000 rows. A selected subsample is students with `fy_gpa` above 3.5: people who already succeeded.

Method of moments: $\hat\mu=\bar x$, $\hat\sigma^2=\frac1n\sum(x_i-\bar x)^2$.

1. $\hat\mu$ and $\hat\sigma^2$ on the full file. You need $\hat\sigma^2$ again in Exercise 7.
2. $n$, $\hat\mu$ and $\hat\sigma^2$ on the selected subsample. Compare the two variances before you move on.
3. Resample the selected rows with replacement at $n=20$, $40$ and $80$, a few thousand draws each, and report the mean of those means and its standard error. Not hashed.

Run (3) before you write the paragraph. The standard error falls as $n$ grows. The gap to the full-file mean does not. More of the same volunteers buys precision around the wrong number.

You can see that gap here only because the file hands you both the class and the subsample. In the situation this models you hold the subsample alone: $n=80$, a clean Normal fit, nothing inside it out of place. That is why this is the dangerous cell.


In [ ]:
from pathlib import Path
import pandas as pd

path = Path("satgpa.csv")
if not path.exists():
    path = "https://vsokolov.org/courses/files/600/satgpa.csv"
sat = pd.read_csv(path)

mu_full = None
sigma2_full = None
n_selected = None
mu_selected = None
sigma2_selected = None
# (3) resampling at n = 20, 40, 80; not hashed
mu_full, sigma2_full, n_selected, mu_selected, sigma2_selected


In [ ]:
check("mu_full", mu_full, "b99aea40fec6914e")
check("sigma2_full", sigma2_full, "846e652d2a66b6d2")
check("n_selected", n_selected, "1a62cb6c20579aaa")
check("mu_selected", mu_selected, "5da8b1e0b41df50b")
check("sigma2_selected", sigma2_selected, "c5cd5813a9585498")
print("ok  exercise 6 sat shift")


The family is still Normal. The rows are not the incoming class. Your resampling in (3) shows the standard error shrinking while the gap to the full-file mean holds. Say what those two numbers separate. Connect this to WHI, Literary Digest, and distribution shift. Four to six sentences.

YOUR ANSWER HERE


## Exercise 7: bad data and a bad model

Same selected subsample (`fy_gpa` above 3.5). Now fit a Poisson by method of moments, $\hat\lambda=\bar x$, to those SAT totals. A Poisson is a count family. `sat_sum` is a 20--80 scale added twice, not a count of events. A Poisson also forces $\mathrm{Var}=\hat\lambda$, so it pins the standard deviation to $\sqrt{\hat\lambda}$ no matter what the rows say.

Two errors are stacked here. Computing two tails cannot separate them, so compute three.

1. $\hat\lambda$ on the selected rows (it should match `mu_selected`).
2. $P(X\ge 140)$ under three fits:
    - `p_ge140_normal`, Normal on the **full file** from Exercise 6. The reference.
    - `p_ge140_normal_sel`, Normal on the **selected** rows. Right family, wrong sample: the data error on its own.
    - `p_ge140_poisson`, Poisson on the **selected** rows. Wrong sample and wrong family.

    $P(X\ge 140)$ includes $140$; check that your tail does not start at $141$.
3. Print two ratios: selected-Normal over full-Normal, then selected-Poisson over selected-Normal. Not hashed.

The two ratios do not point the same way. Work out which error inflated the tail and which one pulled it back before you write the paragraph.

Contrast with Exercise 3, where a *thresholded* SAT was a legitimate binomial.


In [ ]:
from scipy.stats import poisson, norm

lam_selected = None
p_ge140_poisson = None      # Poisson on the selected rows
p_ge140_normal = None       # full-file Normal, P(X >= 140)
p_ge140_normal_sel = None   # Normal fitted to the selected rows
lam_selected, p_ge140_poisson, p_ge140_normal, p_ge140_normal_sel


In [ ]:
check("lam_selected", lam_selected, "980ec9c796af07d3")
check("p_ge140_poisson", p_ge140_poisson, "59495240ef61cc55")
check("p_ge140_normal", p_ge140_normal, "c41cb736b03302a6")
check("p_ge140_normal_sel", p_ge140_normal_sel, "c42e2f89a68070e3")
print("ok  exercise 7 bad bad")


Two mistakes at once: wrong process and wrong law. Use your three numbers: one error inflated the tail, the other pulled it back, so the net gap understates how wrong the fit is. Which is which, and why does a Poisson pin the spread the way it does? Then say why a tail computed on `sat_sum` for students picked by their GPA is not a statement about next year's applicants, and how that differs from the Bernoulli in Exercise 3. Four to six sentences.

YOUR ANSWER HERE
